# Transmission Line Predictive Maintenance — Survival Analysis

Replicating the methodology of **Yang et al. (2022)** *"Prognostic modeling of predictive maintenance with survival analysis for mobile work equipment"*, Scientific Reports 12, 8529.

**Pipeline:**
1. Data loading & validation
2. Exploratory data analysis
3. Kaplan-Meier survival curves (by component, by region)
4. Nelson-Aalen cumulative hazard
5. Kernel hazard estimation (Epanechnikov)
6. Parametric models: Weibull, Exponential, Log-logistic, Log-normal, Generalized-gamma
7. Model selection via AIC and Log-Likelihood Value

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import (
    load_data, validate_data, summary_stats,
    failure_mode_breakdown, covariate_summary,
    COMPONENTS, REGIONS,
    ENV_COLS, HI_COLS,
)
from src.survival_curves import (
    fit_km_by_group, fit_na_by_group,
    km_summary_table, save_km_table,
)
from src.hazard_models import (
    fit_parametric_models, kernel_hazard_by_group,
)
from src.model_selection import (
    build_comparison_table, best_models,
    save_comparison_table, params_table,
)
from src.visualization import (
    plot_event_distribution, plot_duration_boxplot,
    plot_covariate_heatmap, plot_km_by_component,
    plot_km_by_region, plot_na_by_component,
    plot_kernel_hazard, plot_parametric_sf_per_component,
    plot_model_comparison_heatmap, plot_best_model_sf,
)

print('All imports OK.')

---
## Phase 1 — Data Loading & Validation

In [ ]:
raw = load_data()
print(f'Raw shape: {raw.shape}')
raw.head(3)

In [ ]:
df = validate_data(raw)
print(f'Validated shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

---
## Phase 2 — Exploratory Data Analysis

In [ ]:
stats = summary_stats(df)
print('=== Event / Censoring Summary ===')
display(stats)
stats.to_csv('../outputs/tables/summary_stats.csv')

In [ ]:
fm = failure_mode_breakdown(df)
print('=== Failure Mode Breakdown ===')
display(fm)
fm.to_csv('../outputs/tables/failure_mode_breakdown.csv')

In [ ]:
cov_sum = covariate_summary(df)
print('=== Covariate Summary (Events vs. Censored) ===')
display(cov_sum)
cov_sum.to_csv('../outputs/tables/covariate_summary.csv')

In [ ]:
p = plot_event_distribution(df)
print(f'Saved: {p}')

In [ ]:
p = plot_duration_boxplot(df)
print(f'Saved: {p}')

In [ ]:
cov_cols = [c for c in ENV_COLS + HI_COLS if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
p = plot_covariate_heatmap(df, cov_cols)
print(f'Saved: {p}')

---
## Phase 3 — Kaplan-Meier Survival Analysis

In [ ]:
km_fitters = fit_km_by_group(df, group_col='component', groups=COMPONENTS)
p = plot_km_by_component(km_fitters)
print(f'Saved: {p}')

In [ ]:
km_table = km_summary_table(km_fitters)
print('=== Kaplan-Meier Survival Probability at Key Time Points ===')
display(km_table)
save_km_table(km_table, 'km_survival_summary')

In [ ]:
p = plot_km_by_region(df, REGIONS)
print(f'Saved: {p}')

---
## Phase 4 — Nelson-Aalen Cumulative Hazard

In [ ]:
na_fitters = fit_na_by_group(df, group_col='component', groups=COMPONENTS)
p = plot_na_by_component(na_fitters)
print(f'Saved: {p}')

---
## Phase 5 — Kernel Hazard Estimation (Epanechnikov)

In [ ]:
kernel_data = kernel_hazard_by_group(df, group_col='component', groups=COMPONENTS)
p = plot_kernel_hazard(kernel_data)
print(f'Saved: {p}')

---
## Phase 6 — Parametric Survival Models

Models fitted per component:
- Weibull
- Exponential
- Log-logistic
- Log-normal
- Generalized-gamma

In [ ]:
parametric_results = fit_parametric_models(df, group_col='component', groups=COMPONENTS)
for comp, fits in parametric_results.items():
    print(f'{comp}: {[f.model_name for f in fits]} fitted OK')

In [ ]:
p = plot_parametric_sf_per_component(parametric_results, km_fitters)
print(f'Saved: {p}')

---
## Phase 7 — Model Selection (AIC & Log-Likelihood)

In [ ]:
comparison = build_comparison_table(parametric_results)
print('=== Full AIC / LLV Comparison Table ===')
display(comparison)
save_comparison_table(comparison, 'model_comparison')

In [ ]:
best = best_models(comparison)
print('=== Best Model per Component ===')
display(best)
best.to_csv('../outputs/tables/best_models.csv', index=False)

In [ ]:
p_tbl = params_table(parametric_results)
p_tbl.to_csv('../outputs/tables/fitted_parameters.csv', index=False)
print('Fitted parameters saved.')
display(p_tbl)

In [ ]:
p = plot_model_comparison_heatmap(comparison)
print(f'Saved: {p}')

In [ ]:
p = plot_best_model_sf(parametric_results, best, km_fitters)
print(f'Saved: {p}')

---
## Summary

All outputs written to:
- `outputs/figures/` — PNG plots
- `outputs/tables/` — CSV tables

### Figures generated
| File | Description |
|------|-------------|
| eda_event_distribution.png | Failure vs. censored counts by component |
| eda_duration_boxplot.png | Duration distributions by component |
| eda_covariate_heatmap.png | Correlation matrix of covariates |
| km_by_component.png | Kaplan-Meier curves per component |
| km_by_region.png | KM curves per region × component |
| na_cumulative_hazard.png | Nelson-Aalen cumulative hazard |
| kernel_hazard_epanechnikov.png | Kernel hazard (Epanechnikov) |
| parametric_sf_all_components.png | All 5 parametric models vs. KM |
| model_aic_heatmap.png | AIC heatmap (model × component) |
| best_model_sf.png | Best-fit model vs. KM |

### Tables generated
| File | Description |
|------|-------------|
| summary_stats.csv | Event/censoring summary |
| failure_mode_breakdown.csv | Failure counts by mode |
| covariate_summary.csv | Covariate means by event status |
| km_survival_summary.csv | S(t) at key time points |
| model_comparison.csv | Full AIC/LLV table |
| best_models.csv | Best model per component |
| fitted_parameters.csv | Fitted distribution parameters |